# Milestone 1: Dataset Open Parity

Audience:
- Contributors implementing or validating Rust dataset-open parity against the Python oracle.

Prerequisites:
- `uv` and `cargo` installed.
- Repo checked out with Milestone 1 changes.

Goals:
- Regenerate/validate dataset-open fixtures from real Python responses.
- Validate Rust parity against canonical fixture corpus.
- Smoke-test Rust `/healthz` and `POST /dataset/open`.
- Validate CLI/client dataset-open flows against Rust.


## Gate Targets

Expected pass conditions for this notebook run:
1. `cargo test -p lucida-daemon`
2. `uv run pytest tests/test_milestone1_dataset_open_parity.py`
3. `uv run pytest tests/test_dataset_open_rust_integration.py`
4. `uv run pytest`


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import tempfile
from pathlib import Path

import httpx


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'MIGRATION.md').exists():
            return candidate
    raise RuntimeError('Could not find repo root from current working directory.')


NOTEBOOK_CWD = Path.cwd().resolve()
REPO_ROOT = find_repo_root(NOTEBOOK_CWD)
TESTS_ROOT = REPO_ROOT / 'tests'
for entry in (REPO_ROOT, TESTS_ROOT):
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

FIXTURE_PATH = REPO_ROOT / 'tests/parity/fixtures/milestone1/dataset_open_corpus.json'
REPO_ROOT


In [ ]:
def run_cmd(command: str, *, env: dict[str, str] | None = None) -> subprocess.CompletedProcess[str]:
    merged_env = dict(os.environ)
    if env:
        merged_env.update(env)
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        shell=True,
        text=True,
        capture_output=True,
        env=merged_env,
    )
    print(f'$ {command}')
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    print(f'[exit={completed.returncode}]')
    return completed


print(f'Fixture path: {FIXTURE_PATH}')


## 1) Regenerate Dataset-Open Corpus From Python Oracle

Expectation:
- Command exits `0`.
- `tests/parity/fixtures/milestone1/dataset_open_corpus.json` is refreshed from real Python responses.


In [ ]:
regen = run_cmd(
    'LUCIDA_TEST_BACKEND=python LUCIDA_REGEN_DATASET_OPEN_FIXTURES=1 '
    'uv run pytest tests/test_milestone1_dataset_open_parity.py -q'
)
assert regen.returncode == 0, 'Python fixture regeneration failed.'


## 2) Validate Rust Dataset-Open Parity

Expectation:
- Command exits `0`.
- Rust daemon is auto-spawned by pytest fixture and matches canonical corpus.


In [ ]:
rust_parity = run_cmd('uv run pytest tests/test_milestone1_dataset_open_parity.py -q')
assert rust_parity.returncode == 0, 'Rust dataset-open parity test failed.'


## 3) Manual Rust Daemon Health + Dataset-Open Probe

Expectation:
- `/healthz` returns `{"status": "ok"}`.
- `POST /dataset/open` returns `200` and a `dataset_id` starting with `ds_`.


In [ ]:
from parity.data_setup import build_phase1_dataset_uris
from rust_daemon import start_rust_daemon


daemon = start_rust_daemon(repo_root=REPO_ROOT, env=dict(os.environ))
try:
    health = httpx.get(f'{daemon.base_url}/healthz', timeout=10.0)
    print('healthz:', health.status_code, health.json())

    tmp_root = Path(tempfile.mkdtemp(prefix='lucida-m1-nb-'))
    dataset_uris = build_phase1_dataset_uris(tmp_root)
    dataset_open = httpx.post(
        f'{daemon.base_url}/dataset/open',
        json={'schema_version': 1, 'uri': dataset_uris.local_uri},
        timeout=30.0,
    )
    print('dataset_open_status:', dataset_open.status_code)
    payload = dataset_open.json()
    print('dataset_id:', payload['dataset_summary']['dataset_id'])
    print('warnings:', [item['code'] for item in payload.get('warnings', [])])
    assert health.status_code == 200
    assert health.json().get('status') == 'ok'
    assert dataset_open.status_code == 200
    assert payload['dataset_summary']['dataset_id'].startswith('ds_')
finally:
    daemon.stop()


## 4) Rust CLI + Client Dataset-Open Checks

Expectation:
- CLI command exits `0` and returns schema version `1`.
- Python client call succeeds against Rust backend.


In [ ]:
import json

from lucida.client import LucidaClient
from parity.data_setup import build_phase1_dataset_uris
from rust_daemon import start_rust_daemon


daemon = start_rust_daemon(repo_root=REPO_ROOT, env=dict(os.environ))
try:
    tmp_root = Path(tempfile.mkdtemp(prefix='lucida-m1-cli-client-'))
    dataset_uris = build_phase1_dataset_uris(tmp_root)

    cli_env = {'LUCIDA_BACKEND': 'rust', 'LUCIDA_BASE_URL': daemon.base_url}
    cli_result = run_cmd(
        f"uv run lucida dataset open --uri '{dataset_uris.local_uri}' --json",
        env=cli_env,
    )
    assert cli_result.returncode == 0, 'CLI dataset-open failed against Rust daemon.'
    cli_payload = json.loads(cli_result.stdout)
    assert cli_payload['schema_version'] == 1

    with LucidaClient(base_url=daemon.base_url, backend='rust') as client:
        client_response = client.open_dataset(dataset_uris.local_uri)
    print('client dataset_id:', client_response.dataset_summary.dataset_id)
    assert client_response.schema_version == 1
finally:
    daemon.stop()


## 5) Run Milestone 1 Gates

Expectation:
- All commands exit `0`.
- Final `uv run pytest` remains fully green.


In [ ]:
gate_commands = [
    'cargo test -p lucida-daemon',
    'uv run pytest tests/test_milestone1_dataset_open_parity.py -q',
    'uv run pytest tests/test_dataset_open_rust_integration.py -q',
    'uv run pytest -q',
]
gate_results = []
for command in gate_commands:
    result = run_cmd(command)
    gate_results.append((command, result.returncode))
gate_results
